# 03 — Sonuçlar & Analiz
Eğitilmiş modelin performansını analiz eder. `src/train.py` çalıştırıldıktan sonra kullanılır.

In [ ]:
import sys, os
sys.path.insert(0, os.path.abspath('..'))

import torch
import numpy as np
import matplotlib.pyplot as plt
from src.quantum_model import HybridQNN, load_config
from src.evaluate import load_model, get_predictions, confusion_matrix
from src.predict import predict_frame
from camera.mock_camera import MockCamera

config = load_config()
device = torch.device('cpu')

ckpt = config['paths']['checkpoint']
if not os.path.exists(ckpt):
    print('UYARI: Checkpoint bulunamadı. Önce src/train.py çalıştırın.')
else:
    model = load_model(config, device)
    print('Model yüklendi.')

## Karışıklık Matrisi

In [ ]:
from torch.utils.data import DataLoader, Subset
from torchvision import datasets, transforms

transform = transforms.Compose([
    transforms.ToTensor(),
    transforms.Normalize((0.1307,), (0.3081,)),
])
test_ds = datasets.MNIST(config['paths']['data'], train=False, download=True, transform=transform)
test_ds = Subset(test_ds, list(range(config['training']['test_samples'])))
loader  = DataLoader(test_ds, batch_size=64)

preds, labels = get_predictions(model, loader, device)
acc = (preds == labels).mean()
print(f'Test doğruluğu: %{acc*100:.2f}')

cm = confusion_matrix(preds, labels)
fig, ax = plt.subplots(figsize=(8, 6))
im = ax.imshow(cm, cmap='Blues')
plt.colorbar(im)
ax.set_xticks(range(10))
ax.set_yticks(range(10))
ax.set_xlabel('Tahmin')
ax.set_ylabel('Gerçek')
ax.set_title(f'Karışıklık Matrisi — Hibrit QNN (Doğruluk: %{acc*100:.1f})')
for i in range(10):
    for j in range(10):
        ax.text(j, i, str(cm[i][j]), ha='center', va='center',
                color='white' if cm[i][j] > cm.max()/2 else 'black', fontsize=8)
plt.tight_layout()
plt.savefig('../assets/confusion_matrix.png', dpi=150)
plt.show()

## MockCamera ile Canlı Demo Tahmini

In [ ]:
cam = MockCamera(config)
n_demo = 16
fig, axes = plt.subplots(2, 8, figsize=(16, 5))

for i, ax in enumerate(axes.flat):
    frame = cam.read_frame()
    true_label = cam.current_label
    pred, probs, ms = predict_frame(model, frame, device)
    color = 'green' if pred == true_label else 'red'
    ax.imshow(frame, cmap='gray')
    ax.set_title(f'T:{true_label} P:{pred}', color=color, fontsize=9)
    ax.axis('off')

cam.release()
plt.suptitle('MockCamera Tahminleri (Yeşil=Doğru, Kırmızı=Yanlış)')
plt.tight_layout()
plt.savefig('../assets/demo_predictions.png', dpi=150)
plt.show()